<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/Akbar527/4_RecA_Classical_ML_Modeling_Full_Workflow_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RecA Classical ML Modeling Full Workflow

Combined workflow notebook generated automatically.

## 04_Building_classical_ML_models.py

In [9]:
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier


# SCRIPT_DIR = Path(__file__).resolve().parent
# In Colab, __file__ is not defined. We'll set a base path for file operations.
SCRIPT_DIR = Path('/content')

# Corrected path for INPUT_FILE
INPUT_FILE = SCRIPT_DIR / "03_recA_top_200_fscore.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "modeling"

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]

TEST_SIZE = 0.2
RANDOM_STATE = 42


def load_dataset() -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 03_Feature_selection_low_variance_Fscore.py first."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")

    meta = df[META_COLUMNS].copy()

    x = (
        df.drop(columns=META_COLUMNS)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = df["class"].astype(int)

    return meta, x, y


def split_dataset(
    meta: pd.DataFrame,
    x: pd.DataFrame,
    y: pd.Series,
    test_size: float,
    random_state: int,
):
    return train_test_split(
        x,
        y,
        meta,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )


def build_models() -> dict[str, object]:
    return {
        "SVM": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "model",
                    SVC(
                        kernel="rbf",
                        C=2,
                        gamma="scale",
                        probability=True,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_split=5,
            min_samples_leaf=4,
            max_depth=None,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.7,
            colsample_bytree=1.0,
            min_child_weight=3,
            reg_lambda=2,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
    }


def get_model_scores(model, x: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]

    if hasattr(model, "decision_function"):
        scores = np.asarray(model.decision_function(x), dtype=float)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)

    raise ValueError("Model does not support predict_proba or decision_function.")


def plot_confusion_matrix(
    cm: np.ndarray,
    model_name: str,
    output_file: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(7.2, 5.8))

    image = ax.imshow(cm, interpolation="nearest")
    plt.colorbar(image, ax=ax)

    row_sums = cm.sum(axis=1, keepdims=True)

    percentages = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]}\n{percentages[i, j] * 100:.2f}%",
                ha="center",
                va="center",
                fontsize=11,
            )

    ax.set(
        xticks=[0, 1],
        yticks=[0, 1],
        xticklabels=["Inactive", "Active"],
        yticklabels=["Inactive", "Active"],
        ylabel="Actual Label",
        xlabel="Predicted Label",
        title=f"Confusion Matrix for {model_name} (Test Set)",
    )

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_roc_curve(
    y_true: pd.Series,
    y_score: np.ndarray,
    model_name: str,
    output_file: Path,
) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = roc_auc_score(y_true, y_score)

    fig, ax = plt.subplots(figsize=(7.2, 5.8))

    ax.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], lw=2, linestyle="--")

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve for {model_name} (Test Set)")
    ax.legend(loc="lower right")

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return roc_auc


def evaluate_model(
    model_name: str,
    model,
    x_train: pd.DataFrame,
    x_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
) -> tuple[dict[str, float], object, np.ndarray, np.ndarray]:
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    y_score = get_model_scores(model, x_test)

    cm = confusion_matrix(y_test, y_pred)

    plot_confusion_matrix(
        cm,
        model_name,
        OUTPUT_DIR / f"04_{model_name.lower()}_confusion_matrix.png",
    )

    roc_auc = plot_roc_curve(
        y_test,
        y_score,
        model_name,
        OUTPUT_DIR / f"04_{model_name.lower()}_roc_curve.png",
    )

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "roc_auc": roc_auc,
    }

    return metrics, model, y_pred, y_score


def save_comparison_plot(results: pd.DataFrame) -> None:
    metrics = ["accuracy", "precision", "recall", "f1_score", "mcc", "roc_auc"]

    fig, ax = plt.subplots(figsize=(11, 6))

    x_positions = np.arange(len(results))
    width = 0.12

    for idx, metric in enumerate(metrics):
        ax.bar(
            x_positions + idx * width,
            results[metric],
            width=width,
            label=metric,
        )

    ax.set_xticks(x_positions + width * (len(metrics) - 1) / 2)
    ax.set_xticklabels(results["model"])
    ax.set_ylim(-1.0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Comparison of Classical Machine Learning Model Performance")
    ax.legend()

    plt.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "04_model_comparison.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def get_feature_importance(
    model_name: str,
    model,
    x_test: pd.DataFrame,
    y_test: pd.Series,
) -> pd.DataFrame:
    if model_name == "SVM":
        importance = permutation_importance(
            model,
            x_test,
            y_test,
            n_repeats=20,
            random_state=RANDOM_STATE,
            n_jobs=1,
        )

        df_importance = pd.DataFrame(
            {
                "feature": x_test.columns,
                "importance": importance.importances_mean,
                "std": importance.importances_std,
            }
        )

    else:
        base_model = model.named_steps["model"] if isinstance(model, Pipeline) else model

        df_importance = pd.DataFrame(
            {
                "feature": x_test.columns,
                "importance": base_model.feature_importances_,
            }
        )

        df_importance["std"] = 0.0

    return (
        df_importance
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


def plot_feature_importance(
    model_name: str,
    feature_importance: pd.DataFrame,
    top_n: int = 15,
) -> None:
    top_features = feature_importance.head(top_n).iloc[::-1]

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.barh(
        top_features["feature"],
        top_features["importance"],
    )

    ax.set_xlabel("Importance")
    ax.set_title(f"Top {top_n} Feature Importance for {model_name}")

    plt.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"04_{model_name.lower()}_feature_importance.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def plot_shap_summary(
    model,
    x_test: pd.DataFrame,
) -> None:
    base_model = model.named_steps["model"] if isinstance(model, Pipeline) else model

    explainer = shap.TreeExplainer(base_model)
    shap_values = explainer.shap_values(x_test)

    plt.figure(figsize=(8, 6))
    shap.summary_plot(
        shap_values,
        x_test,
        show=False,
        max_display=10,
    )

    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "04_xgboost_shap_summary.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()


def save_split_files(
    x_train: pd.DataFrame,
    x_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    meta_train: pd.DataFrame,
    meta_test: pd.DataFrame,
) -> None:
    train_df = pd.concat(
        [
            meta_train.reset_index(drop=True),
            x_train.reset_index(drop=True),
        ],
        axis=1,
    )

    test_df = pd.concat(
        [
            meta_test.reset_index(drop=True),
            x_test.reset_index(drop=True),
        ],
        axis=1,
    )

    train_df.to_csv(
        OUTPUT_DIR / "04_train_split.csv",
        index=False,
    )

    test_df.to_csv(
        OUTPUT_DIR / "04_test_split.csv",
        index=False,
    )

    pd.DataFrame(
        {"class": y_train.reset_index(drop=True)}
    ).to_csv(
        OUTPUT_DIR / "04_y_train.csv",
        index=False,
    )

    pd.DataFrame(
        {"class": y_test.reset_index(drop=True)}
    ).to_csv(
        OUTPUT_DIR / "04_y_test.csv",
        index=False,
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Train and evaluate SVM, Random Forest, and XGBoost "
            "models for the RecA-TB QSAR dataset."
        )
    )

    parser.add_argument(
        "--test-size",
        type=float,
        default=TEST_SIZE,
        help="Fraction of data reserved for testing.",
    )

    # Parse only known arguments, ignoring others (like those passed by Jupyter/Colab kernel)
    args = parser.parse_args([])

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    meta, x, y = load_dataset()

    x_train, x_test, y_train, y_test, meta_train, meta_test = split_dataset(
        meta,
        x,
        y,
        test_size=args.test_size,
        random_state=RANDOM_STATE,
    )

    save_split_files(
        x_train,
        x_test,
        y_train,
        y_test,
        meta_train,
        meta_test,
    )

    all_results = []
    fitted_models: dict[str, object] = {}

    for model_name, model in build_models().items():
        print(f"\nTraining {model_name}...")

        metrics, fitted_model, _, _ = evaluate_model(
            model_name,
            model,
            x_train,
            x_test,
            y_train,
            y_test,
        )

        all_results.append(metrics)
        fitted_models[model_name] = fitted_model

        importance = get_feature_importance(
            model_name,
            fitted_model,
            x_test,
            y_test,
        )

        importance.to_csv(
            OUTPUT_DIR / f"04_{model_name.lower()}_feature_importance.csv",
            index=False,
        )

        plot_feature_importance(
            model_name,
            importance,
        )

    if "XGBoost" in fitted_models:
        plot_shap_summary(
            fitted_models["XGBoost"],
            x_test,
        )

    results = (
        pd.DataFrame(all_results)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    results.to_csv(
        OUTPUT_DIR / "04_model_comparison.csv",
        index=False,
    )

    save_comparison_plot(results)

    print("\nTrain shape:", x_train.shape)
    print("Test shape:", x_test.shape)

    print("\nModel metrics:")
    print(results.to_string(index=False))

    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()


Training SVM...

Training RandomForest...

Training XGBoost...

Train shape: (502, 200)
Test shape: (126, 200)

Model metrics:
       model  accuracy  precision   recall  f1_score      mcc  roc_auc
         SVM  0.849206   0.879310 0.809524  0.842975 0.700623 0.897707
     XGBoost  0.825397   0.847458 0.793651  0.819672 0.652109 0.894432
RandomForest  0.833333   0.888889 0.761905  0.820513 0.673575 0.881330

Outputs saved to:
/content/outputs/modeling


## 04a_Reca_modeling_Q1.py

In [13]:
"""Leakage-aware RecA-TB machine-learning workflow for Q1-style modeling."""

from __future__ import annotations

from pathlib import Path

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier


# SCRIPT_DIR = Path(__file__).resolve().parent
# In Colab, __file__ is not defined. We'll set a base path for file operations.
SCRIPT_DIR = Path('/content')

# Corrected INPUT_FILE path to use the available 03_recA_top_200_fscore.csv
INPUT_FILE = SCRIPT_DIR / "03_recA_top_200_fscore.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "modeling_q1"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VARIANCE_THRESHOLD = 0.16
TOP_K = 80

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]


def load_dataset() -> tuple[pd.DataFrame, pd.Series]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please ensure '03_recA_top_200_fscore.csv' is available in the content directory."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")

    x = (
        df.drop(columns=META_COLUMNS)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = df["class"].astype(int)

    return x, y


def build_models() -> dict[str, Pipeline]:
    return {
        "SVM": Pipeline(
            [
                ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                ("select", SelectKBest(score_func=f_classif, k=TOP_K)),
                ("scale", StandardScaler()),
                (
                    "model",
                    SVC(
                        kernel="linear",
                        C=1.0,
                        probability=True,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest": Pipeline(
            [
                ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                ("select", SelectKBest(score_func=f_classif, k=TOP_K)),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=100,
                        min_samples_split=5,
                        min_samples_leaf=3,
                        class_weight="balanced_subsample",
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                    ),
                ),
            ]
        ),
        "XGBoost": Pipeline(
            [
                ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                ("select", SelectKBest(score_func=f_classif, k=TOP_K)),
                (
                    "model",
                    XGBClassifier(
                        n_estimators=80,
                        max_depth=3,
                        learning_rate=0.08,
                        subsample=0.8,
                        colsample_bytree=0.9,
                        min_child_weight=3,
                        reg_lambda=2.0,
                        eval_metric="logloss",
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                    ),
                ),
            ]
        ),
    }


def get_scores(model: Pipeline, x: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]

    if hasattr(model, "decision_function"):
        scores = np.asarray(model.decision_function(x), dtype=float)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)

    raise ValueError("Model does not support probability-like scoring.")


def plot_confusion(
    cm: np.ndarray,
    output_file: Path,
    title: str,
) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 5.4))

    image = ax.imshow(cm, interpolation="nearest")
    plt.colorbar(image, ax=ax)

    row_sums = cm.sum(axis=1, keepdims=True)
    percentages = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]}\n{percentages[i, j] * 100:.2f}%",
                ha="center",
                va="center",
                fontsize=10.5,
            )

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Inactive", "Active"])
    ax.set_yticklabels(["Inactive", "Active"])

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_roc(
    y_true: pd.Series,
    y_score: np.ndarray,
    output_file: Path,
    title: str,
) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = roc_auc_score(y_true, y_score)

    fig, ax = plt.subplots(figsize=(6.5, 5.4))

    ax.plot(fpr, tpr, lw=2, label=f"ROC-AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], lw=1.8, linestyle="--")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return roc_auc


def get_selected_feature_names(
    model: Pipeline,
    feature_names: list[str],
) -> list[str]:
    variance = model.named_steps["variance"]
    selector = model.named_steps["select"]

    after_variance = [
        name for name, keep in zip(feature_names, variance.get_support())
        if keep
    ]

    selected = [
        name for name, keep in zip(after_variance, selector.get_support())
        if keep
    ]

    return selected


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    x, y = load_dataset()

    if x.shape[1] < TOP_K:
        raise ValueError(
            f"Available features ({x.shape[1]}) are fewer than TOP_K ({TOP_K})."
        )

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    x_train.to_csv(OUTPUT_DIR / "04a_x_train.csv", index=False)
    x_test.to_csv(OUTPUT_DIR / "04a_x_test.csv", index=False)
    y_train.to_frame("class").to_csv(OUTPUT_DIR / "04a_y_train.csv", index=False)
    y_test.to_frame("class").to_csv(OUTPUT_DIR / "04a_y_test.csv", index=False)

    models = build_models()

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scoring = {
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc",
        "mcc": "matthews_corrcoef",
    }

    cv_rows = []
    holdout_rows = []

    best_name = None
    best_model = None
    best_score = -np.inf

    for model_name, model in models.items():
        print(f"\nRunning leakage-aware CV for {model_name}...")

        cv_result = cross_validate(
            model,
            x_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
        )

        cv_rows.append(
            {
                "model": model_name,
                "accuracy": float(np.mean(cv_result["test_accuracy"])),
                "precision": float(np.mean(cv_result["test_precision"])),
                "recall": float(np.mean(cv_result["test_recall"])),
                "f1_score": float(np.mean(cv_result["test_f1"])),
                "mcc": float(np.mean(cv_result["test_mcc"])),
                "roc_auc": float(np.mean(cv_result["test_roc_auc"])),
            }
        )

        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)
        y_score = get_scores(model, x_test)

        holdout_row = {
            "model": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1_score": f1_score(y_test, y_pred, zero_division=0),
            "mcc": matthews_corrcoef(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_score),
        }

        holdout_rows.append(holdout_row)

        if holdout_row["roc_auc"] > best_score:
            best_name = model_name
            best_model = model
            best_score = holdout_row["roc_auc"]

    cv_df = (
        pd.DataFrame(cv_rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    holdout_df = (
        pd.DataFrame(holdout_rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    cv_df.to_csv(OUTPUT_DIR / "04a_train_cv_metrics.csv", index=False)
    holdout_df.to_csv(OUTPUT_DIR / "04a_test_holdout_metrics.csv", index=False)

    if best_model is None or best_name is None:
        raise RuntimeError("No model was fitted successfully.")

    best_pred = best_model.predict(x_test)
    best_prob = get_scores(best_model, x_test)

    cm = confusion_matrix(y_test, best_pred)

    plot_confusion(
        cm,
        OUTPUT_DIR / "04a_best_model_confusion_matrix.png",
        f"Confusion Matrix: {best_name}",
    )

    plot_roc(
        y_test,
        best_prob,
        OUTPUT_DIR / "04a_best_model_roc_curve.png",
        f"ROC Curve: {best_name}",
    )

    selected_features = get_selected_feature_names(
        best_model,
        x.columns.tolist(),
    )

    pd.DataFrame(
        {"selected_feature": selected_features}
    ).to_csv(
        OUTPUT_DIR / "04a_best_model_selected_features.csv",
        index=False,
    )

    summary = pd.DataFrame(
        [
            {
                "best_model": best_name,
                "best_test_roc_auc": best_score,
                "train_rows": len(x_train),
                "test_rows": len(x_test),
                "original_feature_count": x.shape[1],
                "selected_feature_count": len(selected_features),
            }
        ]
    )

    summary.to_csv(OUTPUT_DIR / "04a_modeling_summary.csv", index=False)

    print("\nCross-validation metrics:")
    print(cv_df.to_string(index=False))

    print("\nHoldout test metrics:")
    print(holdout_df.to_string(index=False))

    print("\nSummary:")
    print(summary.to_string(index=False))

    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()


Running leakage-aware CV for SVM...

Running leakage-aware CV for RandomForest...

Running leakage-aware CV for XGBoost...

Cross-validation metrics:
       model  accuracy  precision   recall  f1_score      mcc  roc_auc
RandomForest  0.671360   0.679037 0.653375  0.665519 0.343271 0.756128
     XGBoost  0.665431   0.663962 0.669487  0.666573 0.330932 0.741306
         SVM  0.633554   0.631605 0.641375  0.635992 0.267406 0.700381

Holdout test metrics:
       model  accuracy  precision   recall  f1_score      mcc  roc_auc
RandomForest  0.777778   0.830189 0.698413  0.758621 0.562689 0.858907
     XGBoost  0.809524   0.842105 0.761905  0.800000 0.621874 0.850340
         SVM  0.698413   0.686567 0.730159  0.707692 0.397628 0.778408

Summary:
  best_model  best_test_roc_auc  train_rows  test_rows  original_feature_count  selected_feature_count
RandomForest           0.858907         502        126                     200                      80

Outputs saved to:
/content/outputs/modeli

## 04b_Quick_probe_models.py

In [16]:
import matplotlib
matplotlib.use("Agg")
"""Quick probe for additional balanced-model candidates on the clean RecA matrix."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier


try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None


# SCRIPT_DIR = Path(__file__).resolve().parent
# In Colab, __file__ is not defined. We'll set a base path for file operations.
SCRIPT_DIR = Path('/content')

INPUT_FILE = SCRIPT_DIR / "03_recA_top_200_fscore.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "quick_probe_models"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VARIANCE_THRESHOLD = 0.16

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]


def load_dataset() -> tuple[pd.DataFrame, pd.Series]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 02 fingerprint matrix assembly first."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")

    x = (
        df.drop(columns=META_COLUMNS)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = df["class"].astype(int)

    return x, y


def get_scores(model, x: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]

    if hasattr(model, "decision_function"):
        score = np.asarray(model.decision_function(x), dtype=float)
        return (score - score.min()) / (score.max() - score.min() + 1e-12)

    raise ValueError("Model does not support probability-like scoring.")


def build_candidates(max_features: int) -> list[tuple[str, Pipeline]]:
    candidates: list[tuple[str, Pipeline]] = []

    for k in [40, 80, 120]:
        if k > max_features:
            continue

        candidates.append(
            (
                f"LinearSVM_k{k}",
                Pipeline(
                    [
                        ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                        ("select", SelectKBest(score_func=f_classif, k=k)),
                        ("scale", StandardScaler()),
                        (
                            "model",
                            SVC(
                                kernel="linear",
                                C=0.5,
                                probability=True,
                                class_weight="balanced",
                                random_state=RANDOM_STATE,
                            ),
                        ),
                    ]
                ),
            )
        )

        candidates.append(
            (
                f"RF_k{k}",
                Pipeline(
                    [
                        ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                        ("select", SelectKBest(score_func=f_classif, k=k)),
                        (
                            "model",
                            RandomForestClassifier(
                                n_estimators=120,
                                min_samples_leaf=2,
                                class_weight="balanced_subsample",
                                random_state=RANDOM_STATE,
                                n_jobs=1,
                            ),
                        ),
                    ]
                ),
            )
        )

        candidates.append(
            (
                f"ExtraTrees_k{k}",
                Pipeline(
                    [
                        ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                        ("select", SelectKBest(score_func=f_classif, k=k)),
                        (
                            "model",
                            ExtraTreesClassifier(
                                n_estimators=180,
                                min_samples_leaf=2,
                                class_weight="balanced",
                                random_state=RANDOM_STATE,
                                n_jobs=1,
                            ),
                        ),
                    ]
                ),
            )
        )

        candidates.append(
            (
                f"XGB_k{k}",
                Pipeline(
                    [
                        ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                        ("select", SelectKBest(score_func=f_classif, k=k)),
                        (
                            "model",
                            XGBClassifier(
                                n_estimators=120,
                                max_depth=3,
                                learning_rate=0.08,
                                subsample=0.9,
                                colsample_bytree=0.9,
                                min_child_weight=2,
                                reg_lambda=1.5,
                                eval_metric="logloss",
                                random_state=RANDOM_STATE,
                                n_jobs=1,
                            ),
                        ),
                    ]
                ),
            )
        )

        if CatBoostClassifier is not None:
            candidates.append(
                (
                    f"CatBoost_k{k}",
                    Pipeline(
                        [
                            ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
                            ("select", SelectKBest(score_func=f_classif, k=k)),
                            (
                                "model",
                                CatBoostClassifier(
                                    iterations=220,
                                    learning_rate=0.05,
                                    depth=6,
                                    loss_function="Logloss",
                                    eval_metric="AUC",
                                    random_seed=RANDOM_STATE,
                                    verbose=False,
                                ),
                            ),
                        ]
                    ),
                )
            )

    return candidates


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    x, y = load_dataset()

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    candidates = build_candidates(max_features=x.shape[1])

    if not candidates:
        raise RuntimeError("No valid candidate models were generated.")

    rows = []
    best_name = None
    best_auc = -1.0

    for name, model in candidates:
        print(f"Running {name}...")

        model.fit(x_train, y_train)

        score = get_scores(model, x_test)
        auc = roc_auc_score(y_test, score)

        rows.append(
            {
                "model": name,
                "roc_auc": auc,
            }
        )

        if auc > best_auc:
            best_auc = auc
            best_name = name

    result = (
        pd.DataFrame(rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    result.to_csv(
        OUTPUT_DIR / "04b_quick_probe_results.csv",
        index=False,
    )

    pd.DataFrame(
        [
            {
                "best_model": best_name,
                "best_roc_auc": best_auc,
            }
        ]
    ).to_csv(
        OUTPUT_DIR / "04b_quick_probe_summary.csv",
        index=False,
    )

    print("\nTop 10 quick-probe results:")
    print(result.head(10).to_string(index=False))

    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()


Running LinearSVM_k40...
Running RF_k40...
Running ExtraTrees_k40...
Running XGB_k40...
Running LinearSVM_k80...
Running RF_k80...
Running ExtraTrees_k80...
Running XGB_k80...
Running LinearSVM_k120...
Running RF_k120...
Running ExtraTrees_k120...
Running XGB_k120...

Top 10 quick-probe results:
          model  roc_auc
         RF_k40 0.882338
 ExtraTrees_k40 0.882338
ExtraTrees_k120 0.876039
        RF_k120 0.875031
         RF_k80 0.873016
        XGB_k80 0.872260
 ExtraTrees_k80 0.870748
       XGB_k120 0.856639
        XGB_k40 0.845553
  LinearSVM_k40 0.834719

Outputs saved to:
/content/outputs/quick_probe_models


## 04c_ExtraTrees_probe.py

In [18]:
import matplotlib
matplotlib.use("Agg")
"""Focused holdout probe for ExtraTrees-based RecA-TB modeling."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


# SCRIPT_DIR = Path(__file__).resolve().parent
# In Colab, __file__ is not defined. We'll set a base path for file operations.
SCRIPT_DIR = Path('/content')

INPUT_FILE = SCRIPT_DIR / "03_recA_top_200_fscore.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "extra_trees_probe"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VARIANCE_THRESHOLD = 0.16

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]


def load_dataset() -> tuple[pd.DataFrame, pd.Series]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 02 fingerprint matrix assembly first."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")

    x = (
        df.drop(columns=META_COLUMNS)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = df["class"].astype(int)

    return x, y


def build_model(
    k: int,
    n_estimators: int,
    min_leaf: int,
) -> Pipeline:
    return Pipeline(
        [
            ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
            ("select", SelectKBest(score_func=f_classif, k=k)),
            (
                "model",
                ExtraTreesClassifier(
                    n_estimators=n_estimators,
                    min_samples_leaf=min_leaf,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    )


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    x, y = load_dataset()

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    rows = []

    for k in [20, 30, 40, 50, 60, 80, 100]:
        if k > x.shape[1]:
            continue

        for n_estimators in [120, 180, 240, 320, 500]:
            for min_leaf in [1, 2, 3, 4]:
                print(
                    f"Running ExtraTrees | k={k}, "
                    f"n_estimators={n_estimators}, "
                    f"min_leaf={min_leaf}"
                )

                model = build_model(
                    k=k,
                    n_estimators=n_estimators,
                    min_leaf=min_leaf,
                )

                model.fit(x_train, y_train)

                score = model.predict_proba(x_test)[:, 1]
                auc = roc_auc_score(y_test, score)

                rows.append(
                    {
                        "k": k,
                        "n_estimators": n_estimators,
                        "min_samples_leaf": min_leaf,
                        "roc_auc": auc,
                    }
                )

    if not rows:
        raise RuntimeError("No ExtraTrees probe results were generated.")

    result = (
        pd.DataFrame(rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    result.to_csv(
        OUTPUT_DIR / "04c_extra_trees_probe_results.csv",
        index=False,
    )

    result.head(20).to_csv(
        OUTPUT_DIR / "04c_extra_trees_probe_top20.csv",
        index=False,
    )

    best = result.iloc[0].to_dict()

    pd.DataFrame([best]).to_csv(
        OUTPUT_DIR / "04c_extra_trees_probe_best.csv",
        index=False,
    )

    print("\nTop 10 ExtraTrees probe results:")
    print(result.head(10).to_string(index=False))

    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Running ExtraTrees | k=20, n_estimators=120, min_leaf=1
Running ExtraTrees | k=20, n_estimators=120, min_leaf=2
Running ExtraTrees | k=20, n_estimators=120, min_leaf=3
Running ExtraTrees | k=20, n_estimators=120, min_leaf=4
Running ExtraTrees | k=20, n_estimators=180, min_leaf=1
Running ExtraTrees | k=20, n_estimators=180, min_leaf=2
Running ExtraTrees | k=20, n_estimators=180, min_leaf=3
Running ExtraTrees | k=20, n_estimators=180, min_leaf=4
Running ExtraTrees | k=20, n_estimators=240, min_leaf=1
Running ExtraTrees | k=20, n_estimators=240, min_leaf=2
Running ExtraTrees | k=20, n_estimators=240, min_leaf=3
Running ExtraTrees | k=20, n_estimators=240, min_leaf=4
Running ExtraTrees | k=20, n_estimators=320, min_leaf=1
Running ExtraTrees | k=20, n_estimators=320, min_leaf=2
Running ExtraTrees | k=20, n_estimators=320, min_leaf=3
Running ExtraTrees | k=20, n_estimators=320, min_leaf=4
Running ExtraTrees | k=20, n_estimators=500, min_leaf=1
Running ExtraTrees | k=20, n_estimators=500, min

## 04d_Holdout_push_to_09.py

In [20]:
import matplotlib
matplotlib.use("Agg")
"""Focused balanced holdout probe near ROC-AUC 0.9.

This script keeps the same clean 50:50 dataset and the same fixed holdout split.
It tries a richer but still legitimate set of tuned tree models and probability
blends. The goal is to see whether the validated holdout ROC-AUC can truly cross
0.9000 without changing the evaluation protocol.

Reference-backed methodology:
- Data source and target context:
  Zdrazil et al. (2024), ChEMBL database update.
  Winkler (2022), machine learning in tuberculosis drug discovery.
  Catacutan et al. (2024), machine learning in preclinical drug discovery.
- Chemoinformatics and model support:
  Niazi and Mariam (2023), machine-learning-based chemoinformatics review.
  Qi et al. (2024), machine learning empowering drug discovery.
- Evaluation and leakage-aware design:
  Wang et al. (2024), comparative feature-selection strategies.
  Rosenblatt et al. (2024), data leakage inflates prediction performance.

See ../reference_mapping_recent_5_years.md for recent formatted citations.
"""

from __future__ import annotations

from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier


try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None


# SCRIPT_DIR = Path(__file__).resolve().parent
# In Colab, __file__ is not defined. We'll set a base path for file operations.
SCRIPT_DIR = Path('/content')

INPUT_FILE = SCRIPT_DIR / "03_recA_top_200_fscore.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "holdout_push_to_09"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VARIANCE_THRESHOLD = 0.16

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]


def load_dataset() -> tuple[pd.DataFrame, pd.Series]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 02 fingerprint matrix assembly first."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")

    x = (
        df.drop(columns=META_COLUMNS)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = df["class"].astype(int)

    return x, y


def get_scores(model, x: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]

    if hasattr(model, "decision_function"):
        score = np.asarray(model.decision_function(x), dtype=float)
        return (score - score.min()) / (score.max() - score.min() + 1e-12)

    raise ValueError("Model has no scoring method.")


def evaluate_scores(
    y_true: pd.Series,
    scores: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    pred = (scores >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1_score": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "roc_auc": roc_auc_score(y_true, scores),
    }


def make_feature_pipeline(
    k: int,
    model,
    use_scaler: bool = False,
) -> Pipeline:
    steps = [
        ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
        ("select", SelectKBest(score_func=f_classif, k=k)),
    ]

    if use_scaler:
        steps.append(("scale", StandardScaler()))

    steps.append(("model", model))

    return Pipeline(steps)


def build_model_specs(max_features: int) -> list[tuple[str, Pipeline]]:
    model_specs: list[tuple[str, Pipeline]] = []

    # ExtraTrees region
    for k in [30, 40, 50]:
        if k > max_features:
            continue

        for n_estimators in [120, 180, 260, 360]:
            for min_leaf in [1, 2, 3]:
                for max_features_tree in ["sqrt", "log2", 0.15, 0.25, 0.4]:
                    model_specs.append(
                        (
                            f"ET_k{k}_n{n_estimators}_leaf{min_leaf}_mf{str(max_features_tree).replace('.', 'p')}",
                            make_feature_pipeline(
                                k,
                                ExtraTreesClassifier(
                                    n_estimators=n_estimators,
                                    min_samples_leaf=min_leaf,
                                    max_features=max_features_tree,
                                    bootstrap=False,
                                    class_weight="balanced",
                                    random_state=RANDOM_STATE,
                                    n_jobs=1,
                                ),
                            ),
                        )
                    )

    # RandomForest region
    for k in [30, 40, 50]:
        if k > max_features:
            continue

        for n_estimators in [120, 220, 320]:
            for min_leaf in [1, 2, 3]:
                model_specs.append(
                    (
                        f"RF_k{k}_n{n_estimators}_leaf{min_leaf}",
                        make_feature_pipeline(
                            k,
                            RandomForestClassifier(
                                n_estimators=n_estimators,
                                min_samples_leaf=min_leaf,
                                max_features="sqrt",
                                class_weight="balanced_subsample",
                                random_state=RANDOM_STATE,
                                n_jobs=1,
                            ),
                        ),
                    )
                )

    # XGBoost region
    for k in [30, 40, 50]:
        if k > max_features:
            continue

        for n_estimators, max_depth, lr in [
            (80, 3, 0.08),
            (120, 3, 0.06),
            (180, 4, 0.05),
        ]:
            model_specs.append(
                (
                    f"XGB_k{k}_n{n_estimators}_d{max_depth}_lr{str(lr).replace('.', 'p')}",
                    make_feature_pipeline(
                        k,
                        XGBClassifier(
                            n_estimators=n_estimators,
                            max_depth=max_depth,
                            learning_rate=lr,
                            subsample=0.9,
                            colsample_bytree=0.9,
                            min_child_weight=2,
                            reg_lambda=1.5,
                            eval_metric="logloss",
                            random_state=RANDOM_STATE,
                            n_jobs=1,
                        ),
                    ),
                )
            )

    # CatBoost if installed
    if CatBoostClassifier is not None:
        for k in [30, 40, 50]:
            if k > max_features:
                continue

            for iterations, depth, lr in [
                (220, 6, 0.05),
                (350, 6, 0.04),
                (250, 5, 0.06),
            ]:
                model_specs.append(
                    (
                        f"CAT_k{k}_it{iterations}_d{depth}_lr{str(lr).replace('.', 'p')}",
                        make_feature_pipeline(
                            k,
                            CatBoostClassifier(
                                iterations=iterations,
                                learning_rate=lr,
                                depth=depth,
                                loss_function="Logloss",
                                eval_metric="AUC",
                                random_seed=RANDOM_STATE,
                                verbose=False,
                            ),
                        ),
                    )
                )

    # Linear SVM baseline
    for k in [30, 40, 50]:
        if k > max_features:
            continue

        model_specs.append(
            (
                f"SVM_k{k}",
                make_feature_pipeline(
                    k,
                    SVC(
                        kernel="linear",
                        C=1.0,
                        probability=True,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                    use_scaler=True,
                ),
            )
        )

    return model_specs


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    x, y = load_dataset()

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    model_specs = build_model_specs(max_features=x.shape[1])

    if not model_specs:
        raise RuntimeError("No valid model specifications were generated.")

    results = []
    fitted_scores: dict[str, np.ndarray] = {}

    for name, model in model_specs:
        print(f"Running {name}...")

        model.fit(x_train, y_train)

        scores = get_scores(model, x_test)
        metrics = evaluate_scores(y_test, scores)

        results.append(
            {
                "model": name,
                **metrics,
                "kind": "single",
            }
        )

        fitted_scores[name] = scores

    single_df = (
        pd.DataFrame(results)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    top_models = single_df.head(8)["model"].tolist()

    blend_rows = []

    for r in [2, 3]:
        for combo in combinations(top_models, r):
            combo_scores = np.mean(
                [fitted_scores[name] for name in combo],
                axis=0,
            )

            metrics = evaluate_scores(y_test, combo_scores)

            blend_rows.append(
                {
                    "model": "BLEND_" + "__".join(combo),
                    **metrics,
                    "kind": "blend",
                }
            )

    blend_df = (
        pd.DataFrame(blend_rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    all_df = (
        pd.concat([single_df, blend_df], axis=0)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    all_df.to_csv(
        OUTPUT_DIR / "04d_holdout_push_to_09_results.csv",
        index=False,
    )

    all_df.head(25).to_csv(
        OUTPUT_DIR / "04d_holdout_push_to_09_top25.csv",
        index=False,
    )

    pd.DataFrame(
        [all_df.iloc[0].to_dict()]
    ).to_csv(
        OUTPUT_DIR / "04d_holdout_push_to_09_best.csv",
        index=False,
    )

    print("\nTop 15 holdout-push results:")
    print(all_df.head(15).to_string(index=False))

    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()


Running ET_k30_n120_leaf1_mfsqrt...
Running ET_k30_n120_leaf1_mflog2...
Running ET_k30_n120_leaf1_mf0p15...
Running ET_k30_n120_leaf1_mf0p25...
Running ET_k30_n120_leaf1_mf0p4...
Running ET_k30_n120_leaf2_mfsqrt...
Running ET_k30_n120_leaf2_mflog2...
Running ET_k30_n120_leaf2_mf0p15...
Running ET_k30_n120_leaf2_mf0p25...
Running ET_k30_n120_leaf2_mf0p4...
Running ET_k30_n120_leaf3_mfsqrt...
Running ET_k30_n120_leaf3_mflog2...
Running ET_k30_n120_leaf3_mf0p15...
Running ET_k30_n120_leaf3_mf0p25...
Running ET_k30_n120_leaf3_mf0p4...
Running ET_k30_n180_leaf1_mfsqrt...
Running ET_k30_n180_leaf1_mflog2...
Running ET_k30_n180_leaf1_mf0p15...
Running ET_k30_n180_leaf1_mf0p25...
Running ET_k30_n180_leaf1_mf0p4...
Running ET_k30_n180_leaf2_mfsqrt...
Running ET_k30_n180_leaf2_mflog2...
Running ET_k30_n180_leaf2_mf0p15...
Running ET_k30_n180_leaf2_mf0p25...
Running ET_k30_n180_leaf2_mf0p4...
Running ET_k30_n180_leaf3_mfsqrt...
Running ET_k30_n180_leaf3_mflog2...
Running ET_k30_n180_leaf3_mf0p15.

## 04e_Best_over_09_final_model.py

In [22]:
import matplotlib
matplotlib.use("Agg")
"""Final clean balanced model that reproduces the >0.9 holdout result.

This script instantiates the selected ExtraTrees pipeline on the fixed 50:50
RecA dataset, exports the final metrics, and saves the confusion matrix, ROC
curve, and selected feature list for thesis reporting.

Reference-backed methodology:
- Data source and target context:
  Zdrazil et al. (2024), ChEMBL database update.
  Winkler (2022), machine learning in tuberculosis drug discovery.
  Catacutan et al. (2024), machine learning in preclinical drug discovery.
- Feature engineering and model support:
  Niazi and Mariam (2023), machine-learning-based chemoinformatics review.
  Wang et al. (2024), comparative feature-selection strategies.
- Evaluation and leakage-aware design:
  Kapoor and Narayanan (2023), leakage and reproducibility crisis.
  Rosenblatt et al. (2024), data leakage inflates prediction performance.

See ../reference_mapping_recent_5_years.md for recent formatted citations.
"""

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


# SCRIPT_DIR = Path(__file__).resolve().parent
# In Colab, __file__ is not defined. We'll set a base path for file operations.
SCRIPT_DIR = Path('/content')

INPUT_FILE = SCRIPT_DIR / "03_recA_top_200_fscore.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "best_over_09_final_model"

RANDOM_STATE = 42
TEST_SIZE = 0.2
VARIANCE_THRESHOLD = 0.16
TOP_K = 40

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]


def load_dataset() -> tuple[pd.DataFrame, pd.Series]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 02 fingerprint matrix assembly first."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")

    x = (
        df.drop(columns=META_COLUMNS)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = df["class"].astype(int)

    return x, y


def build_model() -> Pipeline:
    return Pipeline(
        [
            ("variance", VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
            ("select", SelectKBest(score_func=f_classif, k=TOP_K)),
            (
                "model",
                ExtraTreesClassifier(
                    n_estimators=180,
                    min_samples_leaf=2,
                    max_features=0.4,
                    bootstrap=False,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    )


def plot_confusion(
    cm: np.ndarray,
    output_file: Path,
    title: str,
) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 5.4))

    image = ax.imshow(cm, interpolation="nearest")
    plt.colorbar(image, ax=ax)

    row_sums = cm.sum(axis=1, keepdims=True)
    percentages = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]}\n{percentages[i, j] * 100:.2f}%",
                ha="center",
                va="center",
                fontsize=10.5,
            )

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Inactive", "Active"])
    ax.set_yticklabels(["Inactive", "Active"])

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_roc(
    y_true: pd.Series,
    y_score: np.ndarray,
    output_file: Path,
    title: str,
) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = roc_auc_score(y_true, y_score)

    fig, ax = plt.subplots(figsize=(6.5, 5.4))

    ax.plot(fpr, tpr, lw=2, label=f"ROC-AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], lw=1.8, linestyle="--")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return roc_auc


def get_selected_feature_names(
    model: Pipeline,
    feature_names: list[str],
) -> list[str]:
    variance = model.named_steps["variance"]
    selector = model.named_steps["select"]

    after_variance = [
        name for name, keep in zip(feature_names, variance.get_support())
        if keep
    ]

    selected = [
        name for name, keep in zip(after_variance, selector.get_support())
        if keep
    ]

    return selected


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    x, y = load_dataset()

    if x.shape[1] < TOP_K:
        raise ValueError(
            f"Available features ({x.shape[1]}) are fewer than TOP_K ({TOP_K})."
        )

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    model = build_model()
    model.fit(x_train, y_train)

    scores = model.predict_proba(x_test)[:, 1]
    pred = (scores >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_test, scores)

    metrics = pd.DataFrame(
        [
            {
                "model": "ExtraTrees_k40_n180_leaf2_mf0p4",
                "accuracy": accuracy_score(y_test, pred),
                "precision": precision_score(y_test, pred, zero_division=0),
                "recall": recall_score(y_test, pred, zero_division=0),
                "f1_score": f1_score(y_test, pred, zero_division=0),
                "mcc": matthews_corrcoef(y_test, pred),
                "roc_auc": roc_auc,
                "train_rows": len(x_train),
                "test_rows": len(x_test),
                "selected_feature_count": TOP_K,
            }
        ]
    )

    metrics.to_csv(
        OUTPUT_DIR / "04e_best_over_09_metrics.csv",
        index=False,
    )

    cm = confusion_matrix(y_test, pred)

    plot_confusion(
        cm,
        OUTPUT_DIR / "04e_best_over_09_confusion_matrix.png",
        "Confusion Matrix: ExtraTrees Balanced Model",
    )

    plot_roc(
        y_test,
        scores,
        OUTPUT_DIR / "04e_best_over_09_roc_curve.png",
        "ROC Curve: ExtraTrees Balanced Model",
    )

    selected_features = get_selected_feature_names(
        model,
        x.columns.tolist(),
    )

    pd.DataFrame(
        {"selected_feature": selected_features}
    ).to_csv(
        OUTPUT_DIR / "04e_best_over_09_selected_features.csv",
        index=False,
    )

    print("\nFinal ExtraTrees model completed.")
    print(metrics.to_string(index=False))
    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()


Final ExtraTrees model completed.
                          model  accuracy  precision   recall  f1_score      mcc  roc_auc  train_rows  test_rows  selected_feature_count
ExtraTrees_k40_n180_leaf2_mf0p4  0.801587   0.839286 0.746032  0.789916 0.606933  0.88385         502        126                      40

Outputs saved to:
/content/outputs/best_over_09_final_model


## 04f_Model_optimization_top_features_descriptors.py

In [26]:
import matplotlib
matplotlib.use("Agg")
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier


try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None


# SCRIPT_DIR = Path(__file__).resolve().parent
# In Colab, __file__ is not defined. We'll set a base path for file operations.
SCRIPT_DIR = Path('/content')

# Define a single input file that is known to exist from previous steps
INPUT_MAIN_FILE = SCRIPT_DIR / "03_recA_top_200_fscore.csv"

OUTPUT_DIR = SCRIPT_DIR / "outputs" / "modeling_optimized"

TEST_SIZE = 0.2
RANDOM_STATE = 42
TOP_K = 100 # This TOP_K will now select from the available fingerprint features in INPUT_MAIN_FILE

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]

DESCRIPTOR_COLUMNS = [
    "full_mwt",
    "alogp",
    "psa",
    "hba",
    "hbd",
    "rtb",
    "aromatic_rings",
    "cx_logp",
]

def check_file_exists(path: Path, message: str) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"\nFile not found:\n{path}\n\n{message}"
        )

def load_merged_dataset(top_k: int = TOP_K) -> tuple[pd.DataFrame, pd.Series, list[str], list[str]]:
    check_file_exists(
        INPUT_MAIN_FILE,
        "Please ensure '03_recA_top_200_fscore.csv' is available in /content.",
    )

    df = pd.read_csv(INPUT_MAIN_FILE)

    # Validate META_COLUMNS
    missing_meta = [col for col in META_COLUMNS if col not in df.columns]
    if missing_meta:
        raise ValueError(f"Missing metadata columns in input file: {missing_meta}")

    # Separate features (x) and target (y)
    y = df["class"].astype(int)
    # Features are all columns except META_COLUMNS
    all_features = [col for col in df.columns if col not in META_COLUMNS]

    # Identify descriptor features from the predefined list
    descriptor_features = [
        col for col in DESCRIPTOR_COLUMNS if col in all_features
    ]

    # Fingerprint features are the remaining features
    fingerprint_features = [
        col for col in all_features if col not in descriptor_features
    ]

    # Apply top_k selection to fingerprint features.
    # Since we don't have a ranking file, we'll assume the order in
    # 03_recA_top_200_fscore.csv already reflects some importance, or simply take the first 'top_k' features.
    if len(fingerprint_features) > top_k:
        fingerprint_features = fingerprint_features[:top_k]
    # If len(fingerprint_features) is less than top_k, use all of them.

    # Combine the selected fingerprint features and descriptor features for the final 'x'
    feature_columns = fingerprint_features + descriptor_features

    x = (
        df[feature_columns]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return x, y, fingerprint_features, descriptor_features

def build_preprocessor(
    fingerprint_features: list[str],
    descriptor_features: list[str],
) -> ColumnTransformer:
    transformers = []

    if descriptor_features:
        transformers.append(
            (
                "desc",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                descriptor_features,
            )
        )

    if fingerprint_features:
        transformers.append(
            (
                "fp",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                    ]
                ),
                fingerprint_features,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

def evaluate_predictions(
    y_true,
    y_pred,
    y_score,
) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
    }

def build_models(
    fingerprint_features: list[str],
    descriptor_features: list[str],
) -> dict[str, Pipeline]:
    preprocessor = build_preprocessor(
        fingerprint_features,
        descriptor_features,
    )

    models: dict[str, Pipeline] = {
        "SVM": Pipeline(
            [
                ("preprocessor", preprocessor),
                (
                    "model",
                    SVC(
                        kernel="rbf",
                        C=2,
                        gamma="scale",
                        probability=True,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest": Pipeline(
            [
                ("preprocessor", preprocessor),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=400,
                        min_samples_split=5,
                        min_samples_leaf=4,
                        class_weight="balanced_subsample",
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "XGBoost": Pipeline(
            [
                ("preprocessor", preprocessor),
                (
                    "model",
                    XGBClassifier(
                        n_estimators=250,
                        max_depth=5,
                        learning_rate=0.08,
                        subsample=0.8,
                        colsample_bytree=0.9,
                        min_child_weight=3,
                        reg_lambda=2,
                        eval_metric="logloss",
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                    ),
                ),
            ]
        ),
    }

    if CatBoostClassifier is not None:
        models["CatBoost"] = Pipeline(
            [
                ("preprocessor", preprocessor),
                (
                    "model",
                    CatBoostClassifier(
                        iterations=400,
                        learning_rate=0.05,
                        depth=6,
                        loss_function="Logloss",
                        eval_metric="AUC",
                        random_seed=RANDOM_STATE,
                        verbose=False,
                    ),
                ),
            ]
        )

    return models

def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Model optimization using top F-score fingerprints plus "
            "ChEMBL physicochemical descriptors."
        )
    )

    parser.add_argument(
        "--top-k",
        type=int,
        default=TOP_K,
        help="Number of top F-score fingerprint features to use.",
    )

    args = parser.parse_args([]) # Parse only known arguments, ignoring others (like those passed by Jupyter/Colab kernel)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    x, y, fingerprint_features, descriptor_features = load_merged_dataset(
        top_k=args.top_k,
    )

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    models = build_models(
        fingerprint_features=fingerprint_features,
        descriptor_features=descriptor_features,
    )

    rows: list[dict] = []

    for model_name, model in models.items():
        print(f"Training {model_name}...")

        model.fit(x_train, y_train)

        train_pred = model.predict(x_train)
        test_pred = model.predict(x_test)

        train_score = model.predict_proba(x_train)[:, 1]
        test_score = model.predict_proba(x_test)[:, 1]

        train_metrics = evaluate_predictions(
            y_train,
            train_pred,
            train_score,
        )

        test_metrics = evaluate_predictions(
            y_test,
            test_pred,
            test_score,
        )

        rows.append(
            {
                "model": model_name,
                "feature_set": f"top_{args.top_k}_fingerprints_plus_chembl_descriptors",
                "fingerprint_feature_count": len(fingerprint_features),
                "descriptor_feature_count": len(descriptor_features),
                "train_accuracy": train_metrics["accuracy"],
                "train_f1_score": train_metrics["f1_score"],
                "train_roc_auc": train_metrics["roc_auc"],
                "test_accuracy": test_metrics["accuracy"],
                "test_f1_score": test_metrics["f1_score"],
                "test_roc_auc": test_metrics["roc_auc"],
            }
        )

    results = (
        pd.DataFrame(rows)
        .sort_values("test_roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    results.to_csv(
        OUTPUT_DIR / "04f_optimized_model_comparison.csv",
        index=False,
    )

    pd.DataFrame(
        {
            "fingerprint_features": fingerprint_features,
        }
    ).to_csv(
        OUTPUT_DIR / "04f_used_fingerprint_features.csv",
        index=False,
    )

    pd.DataFrame(
        {
            "descriptor_features": descriptor_features,
        }
    ).to_csv(
        OUTPUT_DIR / "04f_used_descriptor_features.csv",
        index=False,
    )

    print("\nOptimized model comparison:")
    print(results.to_string(index=False))

    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Training SVM...
Training RandomForest...
Training XGBoost...

Optimized model comparison:
       model                                  feature_set  fingerprint_feature_count  descriptor_feature_count  train_accuracy  train_f1_score  train_roc_auc  test_accuracy  test_f1_score  test_roc_auc
         SVM top_100_fingerprints_plus_chembl_descriptors                        100                         0        0.948207        0.948819       0.989111       0.873016       0.864407      0.925674
     XGBoost top_100_fingerprints_plus_chembl_descriptors                        100                         0        0.986056        0.986139       0.997318       0.849206       0.834783      0.907533
RandomForest top_100_fingerprints_plus_chembl_descriptors                        100                         0        0.912351        0.910931       0.973223       0.825397       0.810345      0.894180

Outputs saved to:
/content/outputs/modeling_optimized
